<a href="https://colab.research.google.com/github/jarekwan/praca_inzynierska/blob/main/train_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/ml_project', exist_ok=True)
print("folder ready")

import sys
sys.path.append('/content/drive/MyDrive/ml_project')

Mounted at /content/drive
folder ready


In [2]:
%%writefile /content/drive/MyDrive/ml_project/train_ml.py
# -*- coding: utf-8 -*-
import os
import ast
import pandas as pd
import pickle

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV


target_dir = "/content/drive/MyDrive/ml_project"
method_file = os.path.join(target_dir, "ml_method.txt")
x_train_path = os.path.join(target_dir, "X_train.pkl")
y_train_path = os.path.join(target_dir, "y_train.pkl")
model_output_path = os.path.join(target_dir, "ml_trained.pkl")


# ---------------------------------------------------------
# helper: convert text like "alpha=1,max_depth=5"
#         into python dict: {"alpha":1,"max_depth":5}
# ---------------------------------------------------------
def parse_manual_params(text):
    params = {}
    if text.strip() == "":
        return params
    parts = text.split(",")
    for p in parts:
        if "=" in p:
            k, v = p.split("=")
            k = k.strip()
            v = v.strip()
            try:
                v = ast.literal_eval(v)
            except:
                pass
            params[k] = v
    return params


# ---------------------------------------------------------
# helper: convert text like "alpha=[0.1,1,10]"
# ---------------------------------------------------------
def parse_grid_params(text):
    params = {}
    if text.strip() == "":
        return params
    parts = text.split(",")
    for p in parts:
        if "=" in p:
            k, v = p.split("=")
            k = k.strip()
            v = v.strip()
            try:
                v = ast.literal_eval(v)
            except:
                pass
            params[k] = v
    return params


# ---------------------------------------------------------
# helper: convert ranges like "n_estimators=100-500"
# ---------------------------------------------------------
def parse_random_params(text):
    params = {}
    if text.strip() == "":
        return params
    parts = text.split(",")
    for p in parts:
        if "=" in p and "-" in p:
            k, v = p.split("=")
            k = k.strip()
            lo, hi = v.split("-")
            lo = int(lo)
            hi = int(hi)
            params[k] = range(lo, hi + 1)
    return params


# ---------------------------------------------------------
# train_ml
# ---------------------------------------------------------
def train_ml():

    if not os.path.exists(method_file):
        raise FileNotFoundError("ml_method.txt not found")

    if not os.path.exists(x_train_path):
        raise FileNotFoundError("X_train.pkl not found")

    if not os.path.exists(y_train_path):
        raise FileNotFoundError("y_train.pkl not found")

    # load training data
    x_train = pd.read_pickle(x_train_path)
    y_train = pd.read_pickle(y_train_path)

    # load model + mode + params
    with open(method_file, "r") as f:
        lines = f.read().strip().split("\n")

    model_name = lines[0]
    mode = lines[1]
    raw_params = lines[2]

    # select base model
    if model_name == "linear_regression":
        base_model = LinearRegression()
    elif model_name == "ridge":
        base_model = Ridge()
    elif model_name == "lasso":
        base_model = Lasso()
    elif model_name == "random_forest_regressor":
        base_model = RandomForestRegressor()
    elif model_name == "svr":
        base_model = SVR()
    elif model_name == "xgboost_regressor":
        base_model = XGBRegressor()
    elif model_name == "mlp_regressor":
        base_model = MLPRegressor(max_iter=500)
    else:
        raise ValueError("unknown model type")

    # ---------------------------------------------------------
    # manual mode
    # ---------------------------------------------------------
    if mode == "manual":
        params = parse_manual_params(raw_params)
        model = base_model.set_params(**params)
        model.fit(x_train, y_train)

    # ---------------------------------------------------------
    # grid search
    # ---------------------------------------------------------
    elif mode == "grid_search":
        grid_params = parse_grid_params(raw_params)
        search = GridSearchCV(base_model, grid_params, cv=3)
        search.fit(x_train, y_train)
        model = search.best_estimator_

    # ---------------------------------------------------------
    # random search
    # ---------------------------------------------------------
    elif mode == "random_search":
        random_params = parse_random_params(raw_params)
        search = RandomizedSearchCV(base_model, random_params, cv=3, n_iter=20)
        search.fit(x_train, y_train)
        model = search.best_estimator_

    else:
        raise ValueError("unknown parameter selection mode")

    # save trained model
    with open(model_output_path, "wb") as f:
        pickle.dump(model, f)

    print("saved:", model_output_path)
    print("model training completed")

    return model


Writing /content/drive/MyDrive/ml_project/train_ml.py
